# Quickstart: one untrained CNN on resting EEG

This notebook walks through goal 1 step by step:

1. load the config and the participants
2. look at one EEG window
3. build a CNN with **random weights** (never trained)
4. turn every window into features, average them per participant
5. classify high vs low depression score (BDI) with a logistic-regression probe

Before starting, set the data path in the terminal you launch Jupyter from:
`export UNTRAINED_EEG_DATA=/path/to/ds003478`. The first run preprocesses
and caches all participants (a few minutes).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # this notebook lives in notebooks/
import matplotlib.pyplot as plt
import numpy as np

from untrained_eeg.data import load_config, load_dataset
from untrained_eeg.models import build_model, embed
from untrained_eeg.variability import participant_features, probe, usable

## 1. Config and participants

In [ ]:
cfg = load_config()
participants, windows = load_dataset(cfg)  # builds data/cache/ the first time
participants.head()

In [ ]:
print(participants.label.value_counts().rename({0: "low BDI", 1: "high BDI"}))

## 2. One EEG window

Each window is 60 channels × 2 s at 128 Hz = 256 samples, in microvolts.

In [ ]:
X, cond = windows["sub-001"]
print("windows:", X.shape, "| conditions:", {c: int((cond == c).sum()) for c in ("closed", "open")})

t = np.arange(X.shape[2]) / cfg["sfreq"]
fig, ax = plt.subplots(figsize=(8, 3))
for i, ch in enumerate([0, 29, 59]):  # FP1, C4, O2
    ax.plot(t, X[0, ch] + 40 * i, linewidth=1)
ax.set_xlabel("time (s)"); ax.set_yticks([]); ax.set_title("sub-001, first window, 3 channels")
plt.show()

## 3. A random-weight CNN

The seed fixes the random weights. Change it and the features change.

In [ ]:
model = build_model("cnn1d", n_chans=X.shape[1], n_times=X.shape[2], seed=0)
print(model)
feats = embed(model, X[:5])
print("features for 5 windows:", feats.shape)

## 4. One feature vector per participant (eyes closed)

In [ ]:
people = usable(participants, windows, "closed")
F = participant_features(model, windows, people.participant_id, "closed")
F.shape  # (participants, features)

## 5. Probe: high vs low BDI

Out-of-fold probabilities from 5-fold cross-validation. AUC 0.5 = chance.

In [ ]:
prob, auc, bacc = probe(F, people.label.to_numpy(), cfg)
print(f"AUC = {auc:.3f}, balanced accuracy = {bacc:.3f}")

## Try it

- Change `seed=0` above and re-run: how much does the AUC move? (That is goal 2.)
- Try `"shallow"` or `"eegnet"` instead of `"cnn1d"`, or `"open"` instead of `"closed"`.